# Basics of Mobile Robotics - Project Report

## Filtering

To ensure robust position estimation of the Thymio robot in the environment, we implemented an **Extended Kalman Filter (EKF)** that fuses data from two sources:
- Visual measurements from the camera (position x, y and orientation θ)
- Motor commands and odometry (left and right wheel speeds)

The filter compensates for uncertainties in the process and the measurements and ensures continuous position estimation even when the camera temporarily loses sight of the robot.

### State Model

Le modèle cinématique du robot peut être défini de plusieurs manières différentes. 
1. On peut considérer la position du robot $(x, y, \theta)$ comme states du système et la vitesse comme inputs. Avec $A$ une matrice identité 3x3, $B$ la matrice de contrôle et $u$ les vitesses des rouess.
$$\mathbf{x}_{k+1} = \mathbf{A} \mathbf{x}_{k} + \mathbf{B} \mathbf{u}_{k}$$

2. On peut aussi considérer la vitesse comme un état du système, sans inputs externes et ainsi permettre une estimation de la vitesse du robot en plus de la position. Ceci a pour effet d'améliorer la précision globale du filtre de Kalman. $A$ est donc la matrice de transition d'état 5x5.
$$\mathbf{x}_{k+1} = \mathbf{A} \mathbf{x}_{k}$$


Nous avons choisi de prioriser la deuxième approche en déffinissant les états comme suit :

$$\mathbf{x} = \begin{bmatrix} x \\ y \\ \theta \\ v_L \\ v_R \end{bmatrix}$$

where:
- $x, y$ : robot position in mm
- $\theta$ : robot orientation in radians (positive counterclockwise)
- $v_L, v_R$ : left and right wheel speeds in mm/s

### Kinematic Model

Le modèle cinématique du robot différentiel peut être décrit par les équations suivantes : 

$$\mathbf{x}_{k+1} = g(\mathbf{x}_k) = \begin{bmatrix}
x_k + v_{L_k}\Delta t\frac{\cos(\theta)}{2} + v_{R_k}\Delta t\frac{\cos(\theta)}{2} \\
y_k + v_{L_k}\Delta t\frac{\sin(\theta)}{2} + v_{R_k}\Delta t\frac{\sin(\theta)}{2} \\
\theta_k - \Delta t\frac{v_{L_k}}{L} + \Delta t\frac{v_{R_k}}{L}\\
v_{L_k} \\
v_{R_k}
\end{bmatrix} + \mathbf{W}_{k}$$

$$\mathbf{y}_{k} = \mathbf{C} \mathbf{x}_{k} + \mathbf{V}_{k}$$

where:
- $\Delta t$ : time step between two measurements
- $L = 122.7$ mm : robot wheelbase
- $C$ : observation matrix (identity matrix 5x5)
- $W$ : process noise (Gaussian white noise with covariance $Q$)
- $V$ : measurement noise (Gaussian white noise with covariance $R$)

Ceci se traduit par la matrice de transition d'état non-linéaire suivante :
$$\mathbf{A} = \begin{bmatrix}
1 & 0 & 0 & \frac{\Delta t}{2}\cos(\theta) & \frac{\Delta t}{2}\cos(\theta) \\
0 & 1 & 0 & \frac{\Delta t}{2}\sin(\theta) & \frac{\Delta t}{2}\sin(\theta) \\
0 & 0 & 1 & -\frac{\Delta t}{L} & \frac{\Delta t}{L} \\
0 & 0 & 0 & 1 & 0 \\
0 & 0 & 0 & 0 & 1
\end{bmatrix}$$

On peut reparquer que le modèle est non-linéaire à cause des termes trigonométriques impliquant l'angle $\theta$.
Il est donc nécessaire d'utiliser un filtre de Kalman étendu pour gérer cette non-linéarité. Nous procédons donc à une linéarisation du modèle à chaque step autour de l'état courant pour ensuite pouvoir travailler sur un modèle linéaire approché. Le jacobien est donné par la matrice de transition d'état $J$.

$$\mathbf{J} = \frac{\delta g(\mathbf{x}_k)}{\delta \mathbf{x}_k} = \begin{bmatrix}
1 & 0 & -\sin(\theta)\Delta t\frac{v_L-v_R}{2} & \frac{\Delta t}{2}\cos(\theta) & \frac{\Delta t}{2}\cos(\theta) \\
0 & 1 & \cos(\theta)\Delta t\frac{v_L-v_R}{2} & \frac{\Delta t}{2}\sin(\theta) & \frac{\Delta t}{2}\sin(\theta) \\
0 & 0 & 1 & -\frac{\Delta t}{L} & \frac{\Delta t}{L} \\
0 & 0 & 0 & 1 & 0 \\
0 & 0 & 0 & 0 & 1
\end{bmatrix}$$

Le Jacobien est utilisé dans l'étape de calcule de la covariance de l'état prédit.

### Covariance Matrices
#### Matrix Q - Process Noise
The **Q** matrix represents the uncertainty in the motion model:

$$\mathbf{Q} = \begin{bmatrix}
Qx & 0 & 0 & 0 & 0 \\
0 & Qy & 0 & 0 & 0 \\
0 & 0 & Q\theta & 0 & 0 \\
0 & 0 & 0 & Qv_L & 0 \\
0 & 0 & 0 & 0 & Qv_R
\end{bmatrix}$$

The values reflect:
- $Qx$, $Qy$: Position uncertainty: 0.17 mm² in x and y
- $Q\theta$: Orientation uncertainty: 0.05 rad²
- $Qv_L$, $Qv_R$: Speed uncertainty: 0.43 and 0.45 (mm/s)²

Les variances des états ont été déterminées empiriquement en fonction des permanences observées lors des tests du robot. 
- Pour déterminer les variances $Qx$ et $Qy$ nous avons fais roulé le robot en ligne droite à vitesse constante sur 50cm et avont mesuré l'écart type de la distance parcourue. 
- Pour déterminer les variances $Q\theta$ nous avons fais tourné le robot sur lui-même à vitesse constante pour un angle de 360° et mesuré l'écart type de l'angle obtenu.
- Pour déterminer les variances $Qv_L$ et $Qv_R$ nous avons mesuré l'écart type des vitesses lues par les encodeurs lorsque le robot roulait à vitesse constante.

#### Matrix R - Measurement Noise
The **R** matrix adapts according to camera availability:

**With camera**:
$$\mathbf{R}_{camera} = \begin{bmatrix}
Rx & 0 & 0 & 0 & 0 \\
0 & Ry & 0 & 0 & 0 \\
0 & 0 & R\theta & 0 & 0 \\
0 & 0 & 0 & Rvl & 0 \\
0 & 0 & 0 & 0 & Rvr
\end{bmatrix}$$

The values are set as follows:
- $Rx$, $Ry$: Position uncertainty from camera measurements: e.g., 5 mm² in x and y
- $R\theta$: Orientation uncertainty from camera measurements: e.g., 0.1 rad²
- $Rvl$, $Rvr$: Wheel speed measurement uncertainty: e.g., 0.5 (mm/s)²

These values are also determined based on sensor characteristics and empirical tuning.
- Pour déterminer les variances $Rx$, $Ry$ et $R\theta$ nous avons mesuré l'écart type des positions lues par la caméra avec un robot immobile. Nous avons placé le robot dans un coin de la map pour avoir une distorsion maximale des mesures.
- Pour déterminer les variances $Rvl$ et $Rvr$ nous avons mesuré l'écart type des vitesses mesurées par le robot lors d'un parcours en ligne droite à vitesse constante.


**Without camera**:

When the camera is unavailable, position uncertainties are set to infinity, forcing the filter to ignore these measurements and rely only on odometry.
$$\mathbf{R}_{no\ camera} = \begin{bmatrix}
\infty & 0 & 0 & 0 & 0 \\
0 & \infty & 0 & 0 & 0 \\
0 & 0 & \infty & 0 & 0 \\
0 & 0 & 0 & Rvl & 0 \\
0 & 0 & 0 & 0 & Rvr
\end{bmatrix}$$

### Kalman Filter Algorithm

The filter follows the classical Kalman filter steps at each iteration:

#### 1. Prediction
A Priori state and variance estimate
$$\hat{\mathbf{x}}_{k} = \mathbf{A} \mathbf{x}_{k-1}$$
$$\hat{\mathbf{P}}_{k} = \mathbf{J} \mathbf{P}_{k-1} \mathbf{J}^T + \mathbf{Q}$$

#### 2. Innovation
$$\mathbf{i}_k = \mathbf{y}_k - \mathbf{C} \hat{\mathbf{x}}_{k}$$
$$\mathbf{S}_k = \mathbf{C} \hat{\mathbf{P}}_{k} \mathbf{C}^T + \mathbf{R}$$
where $\mathbf{C} = \mathbf{I}_5$ (identity observation matrix since all state components are measured).

#### 3. Kalman Gain
$$\mathbf{K}_k = \hat{\mathbf{P}}_{k} \mathbf{C}^T \mathbf{S}_k^{-1}$$
#### 4. Estimation Update
A Posteriori state and variance estimate
$$\mathbf{x}_{k} = \hat{\mathbf{x}}_{k} + \mathbf{K}_k \mathbf{i}_k$$
$$\mathbf{P}_{k} = (\mathbf{I} - \mathbf{K}_k \mathbf{C}) \hat{\mathbf{P}}_{k}$$

### Implementation

The `Filter` class encapsulates the Kalman filter implementation. It is initialized with the initial position measured by the camera:

```python
filter = Filter(thymio)  # thymio contains [x, y, theta] and are init with a camera measurement
```

At each iteration, the `update()` method is called with:
- `measured_pos` : position measured by the camera (or `None` if unavailable)
- `Ts` : time elapsed since the last update
- `ls, rs` : measured left and right wheel speeds

The filter returns the estimated position `[x, y, theta]` which is used for robot navigation.